# Neccessities


## Matrix gen

In [3]:
import torch
import math
import numpy as np

class MatrixGenerator:
    def __init__(self, device=None, dtype=torch.float64, seed=42):
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dtype = dtype
        self.seed = seed

    def _set_seed(self):
        """Set random seed for reproducibility."""
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)

    def generate(self, rows, cols, cond=1.0, rank=None):
        """
        Generates a matrix based on specifications.

        Args:
            rows: Number of rows
            cols: Number of columns
            cond: Condition number (default 1.0 for well-conditioned)
            rank: Optional rank for rank-deficient matrices
        """
        self._set_seed()
        r = min(rows, cols) if rank is None else rank

        # Generate random matrices with correct dtype
        rand_U = torch.randn(rows, r, dtype=self.dtype, device=self.device)
        rand_V = torch.randn(cols, r, dtype=self.dtype, device=self.device)

        U, _ = torch.linalg.qr(rand_U, 'complete')
        V, _ = torch.linalg.qr(rand_V, 'complete')

        # Singular values with specified condition number
        if cond == 1.0:
            s = torch.ones(r, dtype=self.dtype, device=self.device)
        else:
            s = torch.logspace(math.log10(1.0), math.log10(1.0 / cond), steps=r, dtype=self.dtype, device=self.device)

        Sigma = torch.zeros((rows, cols), dtype=self.dtype, device=self.device)
        Sigma[:r, :r] = torch.diag(s)

        A = U @ Sigma @ V.T
        return A

    # ========== Square Matrix Families ==========

    def square_well_conditioned(self, n, cond=1.1, sigma_min=0.5):
        """
        Generate a square well-conditioned matrix.

        Args:
            n: Size of the n x n matrix
            cond: Condition number (close to 1 for well-conditioned, default 1.1)
            sigma_min: Minimum singular value (default 0.5)
        """
        self._set_seed()
        A = torch.randn(n, n, dtype=self.dtype, device=self.device)
        # Orthonormalize to make well-conditioned
        Q, _ = torch.linalg.qr(A)
        # Singvals between sigma_min and sigma_min*cond make it well-conditioned
        s = torch.linspace(sigma_min, sigma_min * cond, n, dtype=self.dtype, device=self.device)
        return Q @ torch.diag(s) @ Q.T

    def square_moderately_ill_conditioned(self, n, cond=1e3):
        """
        Generate a square moderately ill-conditioned matrix.
        Condition number ~ 10^3 is moderate for numerical computations.

        Args:
            n: Size of the n x n matrix
            cond: Condition number (default 1000)
        """
        self._set_seed()
        return self.generate(n, n, cond=cond)

    def square_severely_ill_conditioned(self, n, cond=1e8):
        """
        Generate a square severely ill-conditioned matrix.
        Condition number ~ 10^8 is severely ill-conditioned.

        Args:
            n: Size of the n x n matrix
            cond: Condition number (default 100 million)
        """
        self._set_seed()
        return self.generate(n, n, cond=cond)

    # ========== Rectangular Matrix Families ==========

    def tall_full_column_rank(self, m, n, cond=1.0):
        """
        Generate a tall matrix (m > n) with full column rank.

        Args:
            m: Number of rows (m > n)
            n: Number of columns
            cond: Condition number of the n x n core matrix
        """
        self._set_seed()
        assert m > n, "Matrix must be tall (more rows than columns)"

        # Generate a tall matrix with full column rank
        A = torch.randn(m, n, dtype=self.dtype, device=self.device)

        # Orthogonalize columns
        Q, R = torch.linalg.qr(A)

        # Scale by singular values
        if cond > 1.0:
            s = torch.logspace(0, math.log10(1.0 / cond), n, dtype=self.dtype, device=self.device)
        else:
            s = torch.ones(n, dtype=self.dtype, device=self.device)

        return Q @ torch.diag(s)

    def fat_full_row_rank(self, m, n, cond=1.0):
        """
        Generate a fat matrix (m < n) with full row rank.

        Args:
            m: Number of rows
            n: Number of columns (n > m)
            cond: Condition number of the m x m core matrix
        """
        self._set_seed()
        assert n > m, "Matrix must be fat (more columns than rows)"

        # Generate a fat matrix with full row rank
        A = torch.randn(m, n, dtype=self.dtype, device=self.device)

        # QR of A.T gives (n x m) Q and (m x m) R
        # We want A = Q @ R where Q is m x m orthogonal and R is m x n
        Q_full, R = torch.linalg.qr(A.T, 'complete')
        # Q_full is n x n, take first m columns to get n x m
        # Then transpose to get m x n, but we need m x m Q for row space
        Q = Q_full.T[:m, :m]  # m x m orthogonal

        # Scale by singular values
        if cond > 1.0:
            s = torch.logspace(0, math.log10(1.0 / cond), m, dtype=self.dtype, device=self.device)
        else:
            s = torch.ones(m, dtype=self.dtype, device=self.device)

        return Q @ torch.diag(s) @ Q_full.T[:m, :]

    # ========== Rank-Deficient Matrix Families ==========

    def rank_deficient(self, m, n, rank=None, cond=1.0):
        """
        Generate a rank-deficient matrix.

        Args:
            m: Number of rows
            n: Number of columns
            rank: Desired rank (default: min(m, n) - 1)
            cond: Condition number of the non-zero singular values
        """
        self._set_seed()
        r = rank if rank is not None else min(m, n) - 1
        assert r < min(m, n), f"Rank {r} must be less than min(m, n) = {min(m, n)}"

        return self.generate(m, n, cond=cond, rank=r)

    def nearly_rank_deficient(self, m, n, rank=None, eps=1e-12):
        """
        Generate a nearly rank-deficient matrix.
        Has one very small singular value that's nearly zero.

        Args:
            m: Number of rows
            n: Number of columns
            rank: Base rank (default: min(m, n) - 1)
            eps: Very small singular value (default 1e-12)
        """
        self._set_seed()
        r = rank if rank is not None else min(m, n) - 1

        # Generate orthogonal bases with correct dtype
        rand_U = torch.randn(m, r + 1, dtype=self.dtype, device=self.device)
        rand_V = torch.randn(n, r + 1, dtype=self.dtype, device=self.device)

        U, _ = torch.linalg.qr(rand_U, 'complete')
        V, _ = torch.linalg.qr(rand_V, 'complete')

        # Create singular values: most are O(1), one is very small
        s = torch.ones(r + 1, dtype=self.dtype, device=self.device)
        s[-1] = eps  # Nearly zero singular value

        Sigma = torch.zeros((m, n), dtype=self.dtype, device=self.device)
        Sigma[:r + 1, :r + 1] = torch.diag(s)

        return U @ Sigma @ V.T

def print_matrix_info(name, A):
    """Print information about a matrix."""
    m, n = A.shape
    cond_fn = lambda x: np.linalg.cond(x.numpy()) if x.device.type == 'cpu' else torch.linalg.cond(A).item()

    print(f"\n{name}:")
    print(f"  Shape: {m} x {n}")
    print(f"  Dtype: {A.dtype}")
    print(f"  Device: {A.device}")

    # Compute singular values
    if A.device.type == 'cpu':
        s = np.linalg.svd(A.numpy(), compute_uv=False)
    else:
        s = torch.linalg.svdvals(A).cpu().numpy()

    print(f"  Singular values: {s[:5]}{'...' if len(s) > 5 else ''}")
    print(f"  Condition number: {s[0] / s[-1] if s[-1] > 1e-15 else 'inf'}")
    print(f"  Rank (tol=1e-10): {np.sum(s > 1e-10)}")
    print(f"  Frobenius norm: {np.linalg.norm(s):.6f}")



## Pseudoinverse

In [4]:
import torch
import torch.nn as nn
import time
import tracemalloc
from typing import Callable


RESIDUAL_FAILURE_THRESHOLD = 1e6
RESIDUAL_FUNCTIONS = {
    'identity',
    'smaller_identity',
    'left_penrose',
    'penrose_1',
    'right_penrose',
    'penrose_2',
    'combined_penrose',
    'max_penrose',
}


class Pseudoinverse(nn.Module):
    def __init__(
        self,
        device=None,
        dtype=torch.float64,
        method='newton_schulz',
        max_iter=100,
        residual_tolerance=1e-16,
        convergence_threshold=1e-16,
        max_residual=RESIDUAL_FAILURE_THRESHOLD,
        eps=1e-16,
        residual_function: str | Callable[[torch.Tensor, torch.Tensor, int, int], torch.Tensor] = 'identity',
        residual_growth_failure_factor: float | None = None,
        svd_default_runtime_failure_multiplier: float | None = None,
    ):
        """
        Pseudoinverse computation with iterative and SVD-based methods.

        Args:
            device: Torch device
            dtype: Data type
            method: Methods include 'newton_schulz', 'chebyshev', 'mid_point',
                    'homeier', 'esmaeili', 'kaur', 'sheikhi' for iterative methods,
                    and 'svd_default', 'svd_jacobian' for SVD-based methods.
            max_iter: Maximum iterations for iterative methods
            residual_tolerance: Threshold for the selected residual check.
                Set to None to disable residual-based stopping.
            convergence_threshold: Threshold for the relative iterate-delta
                convergence check. Set to None to disable convergence-based
                stopping.
            max_residual: Failure threshold for residual explosion. If the
                selected residual exceeds this value, iteration stops and the
                solve is marked failed. Set to None to disable this check.
            eps: Regularization parameter
            residual_function: Residual used for residual checks and reporting.
                Built-ins:
                    'identity'/'smaller_identity': ||I - XA|| or ||I - AX||
                    'left_penrose'/'penrose_1': ||AXA - A|| / ||A||
                    'right_penrose'/'penrose_2': ||XAX - X|| / ||X||
                    'combined_penrose'/'max_penrose': max(penrose_1, penrose_2)
                A callable may also be supplied with signature (A, X, m, n).
            residual_growth_failure_factor: Failure threshold for residual
                regression. If set, iteration stops and fails when the current
                residual exceeds this factor times the previous residual.
            svd_default_runtime_failure_multiplier: Failure threshold relative
                to svd_default runtime on the same matrix. If set, iteration
                stops and fails after this multiple of the baseline runtime.
        """
        super(Pseudoinverse, self).__init__()
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dtype = dtype
        self.max_iter = max_iter
        self.residual_tolerance = residual_tolerance
        self.convergence_threshold = convergence_threshold
        self.max_residual = max_residual
        self.eps = eps
        self.method = method
        self.residual_function = residual_function
        self.residual_growth_failure_factor = residual_growth_failure_factor
        self.svd_default_runtime_failure_multiplier = svd_default_runtime_failure_multiplier

    @staticmethod
    def _I(matrix_ref: torch.Tensor) -> torch.Tensor:
        dim = min(matrix_ref.shape[-2], matrix_ref.shape[-1])
        return torch.eye(dim, device=matrix_ref.device, dtype=matrix_ref.dtype)

    def _initialize(self, A: torch.Tensor) -> torch.Tensor:
        """
        Initial guess X0 for pseudoinverse iteration.
        """
        AH = A.transpose(-2, -1).conj()

        # X0 = A^H / (||A||_1 ||A||_inf)
        norm1 = torch.linalg.matrix_norm(A, ord=1, dim=(-2, -1))
        norminf = torch.linalg.matrix_norm(A, ord=float("inf"), dim=(-2, -1))
        scale = (norm1 * norminf).clamp_min(self.eps)
        return AH / scale[..., None, None]

    def _identity_residual(self, A: torch.Tensor, X: torch.Tensor, m: int, n: int) -> torch.Tensor:
        """
            Residual based on the smaller-side identity:
            - if m >= n: use ||I_n - X A||
            - else:      use ||I_m - A X||
        """
        if m >= n:
            I = self._I(A)
            R = I - X @ A
        else:
            I = self._I(A)
            R = I - A @ X

        return torch.linalg.matrix_norm(R, dim=(-2, -1))

    def _left_penrose_residual(self, A: torch.Tensor, X: torch.Tensor, m: int, n: int) -> torch.Tensor:
        R = torch.linalg.matrix_norm(A @ X @ A - A, dim=(-2, -1)) / (torch.linalg.matrix_norm(A, dim=(-2, -1)) + self.eps)
        return R

    def _right_penrose_residual(self, A: torch.Tensor, X: torch.Tensor, m: int, n: int) -> torch.Tensor:
        R = torch.linalg.matrix_norm(X @ A @ X - X, dim=(-2, -1)) / (torch.linalg.matrix_norm(X, dim=(-2, -1)) + self.eps)
        return R

    def _combined_penrose_residual(self, A: torch.Tensor, X: torch.Tensor, m: int, n: int) -> torch.Tensor:
        left = self._left_penrose_residual(A, X, m, n)
        right = self._right_penrose_residual(A, X, m, n)
        return torch.maximum(left, right)

    def _residual(self, A: torch.Tensor, X: torch.Tensor, m: int, n: int) -> torch.Tensor:
        if callable(self.residual_function):
            residual = self.residual_function(A, X, m, n)
            if not isinstance(residual, torch.Tensor):
                residual = torch.as_tensor(residual, device=A.device, dtype=A.dtype)
            return residual

        name = str(self.residual_function).lower()
        if name in {'identity', 'smaller_identity'}:
            return self._identity_residual(A, X, m, n)
        if name in {'left_penrose', 'penrose_1'}:
            return self._left_penrose_residual(A, X, m, n)
        if name in {'right_penrose', 'penrose_2'}:
            return self._right_penrose_residual(A, X, m, n)
        if name in {'combined_penrose', 'max_penrose'}:
            return self._combined_penrose_residual(A, X, m, n)

        available = ', '.join(sorted(RESIDUAL_FUNCTIONS))
        raise ValueError(f"Unknown residual_function '{self.residual_function}'. Available: {available}")

    def _residual_name(self) -> str:
        if callable(self.residual_function):
            return getattr(self.residual_function, '__name__', 'callable')
        return str(self.residual_function)

    def _step_newton_schulz(self, A, Xk):
        AX = A @ Xk
        I = self._I(AX)
        return Xk @ (2*I - AX)

    def _step_chebyshev(self, A, Xk):
        AX = A @ Xk
        I = self._I(AX)
        return Xk @ (3*I - AX @ (3*I - AX))

    def _step_mid_point(self, A, Xk):
        XA = Xk @ A
        I = self._I(XA)
        term1 = I - XA
        term2 = 3*I - XA
        return (I - 0.25 * term1 @ (term2 @ term2)) @ Xk

    def _step_homeier(self, A, Xk):
        AX = A @ Xk
        I = self._I(AX)
        R = I - AX
        Inner = 2*I - AX
        return Xk @ (I + 0.5 * R @ (I + Inner @ Inner))

    def _step_esmaeili(self, A, Xk):
        AX = A @ Xk
        I = self._I(AX)
        res = 5 * AX - 21 * I
        res = res @ AX + 34 * I
        res = res @ AX - 26 * I
        res = res @ AX + 9 * I
        return Xk @ res

    def _step_kaur(self, A, Xk):
        S = A @ Xk
        I = self._I(S)
        Y = S @ S
        V = 5 * (I - S)
        return Xk @ (V - 5*S + Y @ (5*I + V + Y))

    def _step_sheikhi(self, A, Xk):
        S = A @ Xk
        I = self._I(S)
        inner_S = 7*S - 16*I
        Y = S @ inner_S + 11*I
        return 0.25 * Xk @ (4*I - Y @ S) @ Y

    def _track_memory(self):
        """Get current memory usage in bytes."""
        if self.device.type == 'cuda' and torch.cuda.is_available():
            return torch.cuda.memory_allocated(self.device)
        else:
            process = None
            try:
                import psutil
                process = psutil.Process()
                return process.memory_info().rss
            except ImportError:
                return 0

    # ========== SVD-based Pseudoinverse Methods ==========

    def _svd_pinv(self, A):
        """
        Compute pseudoinverse using full SVD.

        A^+ = V @ diag(1/sigma) @ U^T
        """
        U, s, Vh = torch.linalg.svd(A, full_matrices=False)
        # Invert singular values (with regularization)
        s_inv = torch.where(s > self.eps, 1.0 / s, 0.0)
        # Build pseudoinverse: V @ diag(s_inv) @ U^H
        X = Vh.mH @ torch.diag(s_inv) @ U.mH
        return X

    def _jacobian_svd_pinv(self, A):
        """
        Compute pseudoinverse using Jacobi SVD (one-sided variant).

        Uses the built-in torch.linalg.svd with 'jacobian' driver for the
        one-sided Jacobi method, which is often more accurate for ill-conditioned
        matrices than the default gesdd driver.

        Reference: LAPACK's GESVJ - one-sided Jacobi SVD with convergence control.
        """
        # Use torch's built-in JACOBI driver for SVD
        try:
            U, s, Vh = torch.linalg.svd(A, full_matrices=False, driver='gesvdj')
        except (RuntimeError, ValueError):
            # Fallback to default driver if jacobi is not available or fails
            U, s, Vh = torch.linalg.svd(A, full_matrices=False)

        # Invert singular values (with regularization)
        s_inv = torch.where(s > self.eps, 1.0 / s, 0.0)

        # Build pseudoinverse: V @ diag(1/sigma) @ U^H
        X = Vh.mH @ torch.diag(s_inv) @ U.mH
        return X

    def _select_svd_method(self):
        """Select and execute the appropriate SVD method."""
        if self.method == 'svd_default':
            return self._svd_pinv
        elif self.method == 'svd_jacobian':
            return self._jacobian_svd_pinv
        else:
            raise ValueError(f"Unknown SVD method: {self.method}. Use 'svd_default' or 'svd_jacobian'")

    def _svd_default_runtime(self, A: torch.Tensor) -> float:
        if A.device.type == 'cuda' and torch.cuda.is_available():
            torch.cuda.synchronize(A.device)
        start_time = time.time()
        self._svd_pinv(A)
        if A.device.type == 'cuda' and torch.cuda.is_available():
            torch.cuda.synchronize(A.device)
        return time.time() - start_time

    def _svd_forward(self, A, return_info=True):
        """
        Forward pass using SVD-based pseudoinverse.
        """
        start_time = time.time()
        tracemalloc.start()
        mem_before = self._track_memory()

        svd_fn = self._select_svd_method()
        X = svd_fn(A)

        # Compute final residual
        m, n = A.shape[-2], A.shape[-1]
        final_residual = self._residual(A, X, m, n)
        max_final_residual = torch.max(final_residual).item()

        # Determine success (SVD is a direct method - always "converged")
        converged = True
        stagnated = False

        current_mem = self._track_memory()
        _, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end_time = time.time()

        memory_bytes = max(peak_mem, current_mem - mem_before)

        info = {
            'pinv': X,
            'iterations': 1,  # SVD is a direct method
            'final_residual': max_final_residual,
            'final_convergence_delta': 0.0,
            'residual_function': self._residual_name(),
            'residual_tolerance': self.residual_tolerance,
            'convergence_threshold': self.convergence_threshold,
            'max_residual': self.max_residual,
            'stopped_by': 'direct',
            'success': converged,
            'stagnated': stagnated,
            'failed': False,
            'residual_history': [],
            'memory_bytes': memory_bytes,
            'time_seconds': end_time - start_time
        }

        if return_info:
            return info
        else:
            return X

    def forward(self, A, return_info=True):
        """
        Compute pseudoinverse with runtime statistics.

        Args:
            A: Input matrix
            return_info: If True, returns dict with info; if False, returns just pseudoinverse

        Returns:
            If return_info=True: dict with runtime information
            If return_info=False: just the pseudoinverse tensor
        """
        # Check if using SVD method
        if self.method in ['svd_default', 'svd_jacobian']:
            return self._svd_forward(A, return_info=return_info)
        else:
            return self._iterative_forward(A, return_info=return_info)

    def _iterative_forward(self, A, return_info=True):
        """
        Forward pass using iterative pseudoinverse methods.
        """
        m, n = A.shape[-2], A.shape[-1]
        step_fn = getattr(self, f"_step_{self.method}")

        start_time = time.time()
        tracemalloc.start()
        mem_before = self._track_memory()

        X = self._initialize(A)
        max_final_residual = float('inf')
        max_final_convergence = float('inf')
        failed = False
        stopped_by = ''
        k = -1
        residual_history = [float(torch.max(self._residual(A, X, m, n)).item())] if return_info else []
        convergence_history = []
        last_residual = residual_history[0] if residual_history else float(torch.max(self._residual(A, X, m, n)).item())
        svd_default_time_seconds = None
        runtime_failure_seconds = None
        if self.svd_default_runtime_failure_multiplier is not None:
            svd_default_time_seconds = self._svd_default_runtime(A)
            runtime_failure_seconds = self.svd_default_runtime_failure_multiplier * svd_default_time_seconds
            tracemalloc.stop()
            tracemalloc.start()
            mem_before = self._track_memory()
            start_time = time.time()

        for k in range(self.max_iter):
            X_prev = X
            X = step_fn(A, X_prev)
            err = self._residual(A, X, m, n)
            max_err = torch.max(err)
            max_final_residual = max_err.item()
            convergence = torch.linalg.matrix_norm(X - X_prev, ord='fro', dim=(-2, -1)) / (
                1.0 + torch.linalg.matrix_norm(X_prev, ord='fro', dim=(-2, -1))
            ).clamp_min(self.eps)
            max_convergence = torch.max(convergence)
            max_final_convergence = max_convergence.item()
            if return_info:
                residual_history.append(max_final_residual)
                convergence_history.append(max_final_convergence)

            if (
                not torch.isfinite(max_err)
                or not torch.isfinite(max_convergence)
            ):
                failed = True
                stopped_by = 'non_finite'
                break

            if self.max_residual is not None and max_final_residual > self.max_residual:
                failed = True
                stopped_by = 'max_residual'
                break

            if (
                self.residual_growth_failure_factor is not None
                and max_final_residual > self.residual_growth_failure_factor * last_residual
            ):
                failed = True
                stopped_by = 'residual_growth'
                break

            if A.device.type == 'cuda' and torch.cuda.is_available():
                torch.cuda.synchronize(A.device)
            if runtime_failure_seconds is not None and time.time() - start_time > runtime_failure_seconds:
                failed = True
                stopped_by = 'svd_default_runtime_multiplier'
                break

            residual_reached = (
                self.residual_tolerance is not None
                and max_final_residual < self.residual_tolerance
            )
            convergence_reached = (
                self.convergence_threshold is not None
                and max_final_convergence < self.convergence_threshold
            )
            if residual_reached or convergence_reached:
                if residual_reached and convergence_reached:
                    stopped_by = 'residual_and_convergence'
                elif residual_reached:
                    stopped_by = 'residual'
                else:
                    stopped_by = 'convergence'
                break

            last_residual = max_final_residual

        current_mem = self._track_memory()
        _, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end_time = time.time()

        if k < 0:
            final_residual = self._residual(A, X, m, n)
            max_final_residual = torch.max(final_residual).item()
            max_final_convergence = 0.0

        residual_reached = (
            self.residual_tolerance is not None
            and max_final_residual < self.residual_tolerance
        )
        convergence_reached = (
            self.convergence_threshold is not None
            and max_final_convergence < self.convergence_threshold
        )
        converged = False if failed else (residual_reached or convergence_reached)
        if not stopped_by and converged:
            if residual_reached and convergence_reached:
                stopped_by = 'residual_and_convergence'
            elif residual_reached:
                stopped_by = 'residual'
            else:
                stopped_by = 'convergence'
        stagnated = not converged and not failed and self.max_iter > 0 and k == self.max_iter - 1

        memory_bytes = max(peak_mem, current_mem - mem_before)

        info = {
            'pinv': X,
            'iterations': k + 1 if k >= 0 else 0,
            'final_residual': max_final_residual,
            'final_convergence_delta': max_final_convergence,
            'residual_function': self._residual_name(),
            'residual_tolerance': self.residual_tolerance,
            'convergence_threshold': self.convergence_threshold,
            'max_residual': self.max_residual,
            'residual_growth_failure_factor': self.residual_growth_failure_factor,
            'svd_default_runtime_failure_multiplier': self.svd_default_runtime_failure_multiplier,
            'svd_default_time_seconds': svd_default_time_seconds,
            'runtime_failure_seconds': runtime_failure_seconds,
            'stopped_by': stopped_by,
            'success': converged,
            'stagnated': stagnated,
            'failed': failed,
            'residual_history': residual_history,
            'convergence_history': convergence_history,
            'memory_bytes': memory_bytes,
            'time_seconds': end_time - start_time
        }

        if return_info:
            return info
        else:
            return X

    def pinv(self, A, return_info=True):
        """
        Compute pseudoinverse with runtime statistics.
        """
        return self.forward(A, return_info=return_info)
    


## ELM

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional



class ExtremeLearningMachine(nn.Module):
    """
    Extreme Learning Machine (ELM) with flexible pseudoinverse methods and activation functions.

    The ELM is a single-layer feedforward neural network where:
    - Input weights and biases are randomly assigned and fixed
    - Output weights are solved using pseudoinverse computation
    - Hidden layer activation can be configured

    Pseudoinverse methods available (from pinv.py):
        Iterative: 'newton_schulz', 'chebyshev', 'mid_point',
                   'homeier', 'esmaeili', 'kaur', 'sheikhi'
        SVD-based: 'svd_default', 'svd_jacobian'
    """
    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 device=None,
                 dtype=torch.float64,
                  pinv_method: str = 'svd_default',
                  pinv_kwargs: dict = None,
                 activation: str = 'relu'):
        """
        Initialize the Extreme Learning Machine.

        Args:
            input_size: Number of input features
            hidden_size: Number of hidden neurons
            device: Torch device (cpu or cuda)
            dtype: Data type for computations
            pinv_method: Pseudoinverse method to use. Options from pinv.py:
                Iterative: 'newton_schulz', 'chebyshev', 'mid_point',
                           'homeier', 'esmaeili', 'kaur', 'sheikhi'
                SVD-based: 'svd_default', 'svd_jacobian'
            pinv_kwargs: Additional kwargs for Pseudoinverse class
            activation: Hidden layer activation function ('relu', 'elu', 'tanh', 'sigmoid')
        """
        super(ExtremeLearningMachine, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dtype = dtype
        self.pinv_method = pinv_method
        self.pinv_kwargs = pinv_kwargs or {}
        self.activation = activation.lower()

        # Initialize random hidden layer parameters (fixed, not trained)
        self._init_hidden_layer()

        # Initialize pseudoinverse solver
        self.pinv = Pseudoinverse(
            device=self.device,
            dtype=self.dtype,
            method=self.pinv_method,
            **self.pinv_kwargs
        )

        # Output weights (will be computed, not trained)
        self.output_weights = None

        # Get activation function
        self._activation_fn = self._get_activation_fn(self.activation)

    def _init_hidden_layer(self):
        """Initialize random input weights and biases."""
        # Glorot-style initialization for hidden layer
        gain = torch.sqrt(torch.tensor(2.0 / (self.input_size + self.hidden_size)))

        # Input weights: shape (hidden_size, input_size)
        self.input_weights = nn.Parameter(
            gain * torch.randn(self.hidden_size, self.input_size,
                              device=self.device, dtype=self.dtype),
            requires_grad=False
        )

        # Biases: shape (hidden_size,)
        self.biases = nn.Parameter(
            torch.zeros(self.hidden_size, device=self.device, dtype=self.dtype),
            requires_grad=False
        )

    def _get_activation_fn(self, activation: str):
        """Get activation function by name."""
        activations = {
            'relu': F.relu,
            'elu': F.elu,
            'tanh': F.tanh,
            'sigmoid': F.sigmoid,
        }
        if activation.lower() not in activations:
            raise ValueError(f"Unknown activation '{activation}'. Available: {list(activations.keys())}")
        return activations[activation.lower()]

    def _compute_hidden_layer(self, X: torch.Tensor) -> torch.Tensor:
        """
        Compute hidden layer activations with configurable activation function.

        Args:
            X: Input tensor of shape (batch_size, input_size)

        Returns:
            Hidden activations of shape (batch_size, hidden_size)
        """
        # H = G(X @ W^T + b)
        # where G is the activation function
        H = X @ self.input_weights.t() + self.biases
        H = self._activation_fn(H)
        return H

    def forward(self, X: torch.Tensor, return_info: bool = False) -> torch.Tensor:
        """
        Forward pass through the ELM.

        Args:
            X: Input tensor of shape (batch_size, input_size)
            return_info: If True, returns output dict with pinv info

        Returns:
            Output tensor of shape (batch_size, output_size)
        """
        if self.output_weights is None:
            raise RuntimeError("Model must be trained before forward pass.")

        H = self._compute_hidden_layer(X)
        output = H @ self.output_weights

        if return_info:
            return {'output': output}
        return output

    def train(self, X: Optional[torch.Tensor] = None, y: Optional[torch.Tensor] = None):
        """
        Train mode toggle or fit the ELM output weights.

        This class is an nn.Module, so PyTorch calls train(mode=True) from
        parent modules and eval(). When X and y are tensors, this method keeps
        the original ELM behavior and computes output weights by pseudoinverse.

        Args:
            X: Training inputs of shape (n_samples, input_size)
            y: Training targets of shape (n_samples, output_size)

        Returns:
            self for PyTorch mode toggles, or training info for ELM fitting
        """
        if isinstance(X, bool) and y is None:
            return super().train(X)
        if X is None and y is None:
            return super().train(True)
        if X is None or y is None:
            raise TypeError("ExtremeLearningMachine.train() requires both X and y when fitting.")

        # Ensure correct shapes
        if X.dim() == 1:
            X = X.unsqueeze(0)
        if y.dim() == 1:
            y = y.unsqueeze(0)

        n_samples, n_features = X.shape
        if n_features != self.input_size:
            raise ValueError(f"Expected input size {self.input_size}, got {n_features}")

        # Compute hidden layer activations
        H = self._compute_hidden_layer(X)  # (n_samples, hidden_size)

        # Compute output weights using pseudoinverse:
        # W_out = H^+ @ Y
        pinv_result = self.pinv.pinv(H, return_info=True)
        self.output_weights = pinv_result['pinv'] @ y

        # training info
        info = {
            'n_samples': n_samples,
            'hidden_size': self.hidden_size,
            **{k: v for k, v in pinv_result.items() if k != 'pinv'}
        }

        return info

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        """
        Predict outputs for input X.

        Args:
            X: Input tensor of shape (batch_size, input_size)

        Returns:
            Predicted outputs of shape (batch_size, output_size)
        """
        return self.forward(X)

    def save(self, path: str):
        """Save the model to a file."""
        state = {
            'input_size': self.input_size,
            'hidden_size': self.hidden_size,
            'device': self.device,
            'dtype': self.dtype,
            'pinv_method': self.pinv_method,
            'pinv_kwargs': self.pinv_kwargs,
            'activation': self.activation,
            'input_weights': self.input_weights.cpu(),
            'biases': self.biases.cpu(),
            'output_weights': self.output_weights.cpu() if self.output_weights is not None else None
        }
        torch.save(state, path)

    @classmethod
    def load(cls, path: str) -> 'ExtremeLearningMachine':
        """Load a model from a file."""
        state = torch.load(path, map_location='cpu')

        model = cls(
            input_size=state['input_size'],
            hidden_size=state['hidden_size'],
            device=state['device'],
            dtype=state['dtype'],
            pinv_method=state['pinv_method'],
            pinv_kwargs=state.get('pinv_kwargs', {}),
            activation=state.get('activation', 'relu')
        )

        model.input_weights = nn.Parameter(state['input_weights'], requires_grad=False)
        model.biases = nn.Parameter(state['biases'], requires_grad=False)
        if state['output_weights'] is not None:
            model.output_weights = nn.Parameter(state['output_weights'], requires_grad=False)

        return model



## Metrics

In [6]:
from __future__ import annotations

from typing import Sequence

import numpy as np
import torch


def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    matrix = np.zeros((n_classes, n_classes), dtype=np.int64)
    for true_label, pred_label in zip(y_true, y_pred):
        matrix[int(true_label), int(pred_label)] += 1
    return matrix


def per_class_metrics(
    cm: np.ndarray,
    labels: Sequence[str] | None = None,
) -> list[dict[str, float | int | str]]:
    rows: list[dict[str, float | int | str]] = []
    class_labels = labels if labels is not None else [str(idx) for idx in range(cm.shape[0])]

    for idx, label in enumerate(class_labels):
        tp = int(cm[idx, idx])
        fp = int(cm[:, idx].sum() - tp)
        fn = int(cm[idx, :].sum() - tp)
        support = int(cm[idx, :].sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append(
            {
                "class": label,
                "support": support,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

    return rows


def aggregate_classification_metrics(
    cm: np.ndarray,
    metric_rows: list[dict[str, float | int | str]] | None = None,
) -> dict[str, float]:
    if metric_rows is None:
        metric_rows = per_class_metrics(cm)

    tp = int(np.trace(cm))
    fp = int(cm.sum(axis=0).sum() - tp)
    fn = int(cm.sum(axis=1).sum() - tp)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    macro_f1 = (
        sum(float(row["f1"]) for row in metric_rows) / len(metric_rows)
        if metric_rows
        else 0.0
    )
    recalls = [
        cm[idx, idx] / cm[idx, :].sum() if cm[idx, :].sum() else 0.0
        for idx in range(cm.shape[0])
    ]
    balanced_accuracy = float(np.mean(recalls)) if recalls else 0.0

    total = cm.sum()
    pred_totals = cm.sum(axis=0)
    true_totals = cm.sum(axis=1)
    numerator = np.trace(cm) * total - np.dot(pred_totals, true_totals)
    denominator = np.sqrt(
        (total**2 - np.dot(pred_totals, pred_totals))
        * (total**2 - np.dot(true_totals, true_totals))
    )
    mcc = float(numerator / denominator) if denominator else 0.0

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_accuracy,
        "mcc": mcc,
    }


def classification_metrics_from_scores(
    scores: torch.Tensor,
    labels: torch.Tensor,
    n_classes: int,
) -> dict[str, float]:
    preds = scores.argmax(dim=1).detach().cpu().numpy()
    true = labels.detach().cpu().numpy()
    cm = confusion_matrix(true, preds, n_classes)
    return aggregate_classification_metrics(cm)


def accuracy_from_scores(scores: torch.Tensor, labels: torch.Tensor) -> float:
    return (scores.argmax(dim=1) == labels).to(torch.float64).mean().item()


def mse_from_scores(scores: torch.Tensor, targets_onehot: torch.Tensor) -> float:
    return torch.mean((targets_onehot - scores) ** 2).item()


def matrix_condition_number(matrix: torch.Tensor) -> float:
    with torch.no_grad():
        singular_values = torch.linalg.svdvals(matrix)

    if singular_values.numel() == 0:
        return float("nan")
    sigma_max = singular_values[0].item()
    sigma_min = singular_values[-1].item()
    if sigma_min <= 0.0:
        return float("inf")
    return sigma_max / sigma_min


def evaluate_score_pair(
    train_scores: torch.Tensor,
    test_scores: torch.Tensor,
    y_train: torch.Tensor,
    y_test: torch.Tensor,
    y_train_onehot: torch.Tensor,
    y_test_onehot: torch.Tensor,
) -> dict[str, float]:
    train_metrics = classification_metrics_from_scores(
        train_scores,
        y_train,
        y_train_onehot.shape[1],
    )
    test_metrics = classification_metrics_from_scores(
        test_scores,
        y_test,
        y_test_onehot.shape[1],
    )
    return {
        "train_accuracy": accuracy_from_scores(train_scores, y_train),
        "test_accuracy": accuracy_from_scores(test_scores, y_test),
        "train_mse": mse_from_scores(train_scores, y_train_onehot),
        "test_mse": mse_from_scores(test_scores, y_test_onehot),
        "train_micro_f1": train_metrics["micro_f1"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_mcc": train_metrics["mcc"],
        "test_micro_f1": test_metrics["micro_f1"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_mcc": test_metrics["mcc"],
    }


# Thesis

## Config

In [7]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import List

import torch


PINV_METHODS = [
    "svd_default",
    "svd_jacobian",
    "newton_schulz",
    "chebyshev",
    # "mid_point",
    "homeier",
    "esmaeili",
    "kaur",
    "sheikhi",
]

SVD_METHODS = {"svd_default", "svd_jacobian"}

MATRIX_SIZES = [
    (100, 100),
    (500, 500),
    (500, 200),
    (200, 500),
    (1000, 300),
    (300, 1000),
]


@dataclass
class SyntheticMatrixBenchmarkConfig:
    results_dir: Path = Path("results")
    output_csv: Path = Path("results/synthetic_matrix_benchmark.csv")
    summary_csv: Path = Path("results/synthetic_matrix_benchmark_summary.csv")
    residual_trace_csv: Path = Path("results/synthetic_matrix_residual_traces.csv")
    spectra: List[str] = field(
        default_factory=lambda: [
            "well_conditioned",
            "moderately_ill_conditioned",
            "severely_ill_conditioned",
            "rank_deficient",
            "clustered_small",
        ]
    )
    matrix_sizes: List[tuple[int, int]] = field(default_factory=lambda: MATRIX_SIZES.copy())
    methods: List[str] = field(default_factory=lambda: ["svd_default", "svd_jacobian", "newton_schulz", "kaur", "sheikhi"])
    seeds: List[int] = field(default_factory=lambda: [0, 1, 2])
    target_cols: int = 8
    rank_deficient_zero_fraction: float = 0.1
    small_cluster_fraction: float = 0.1
    pinv_max_iter: int = 50
    pinv_residual_tolerance: float = 1e-12
    pinv_convergence_threshold: float = 1e-12
    pinv_max_residual: float | None = 1e6
    pinv_residual_function: str = "identity"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype: torch.dtype = torch.float64
    verbose: bool = True


@dataclass
class DatasetExperimentConfig:
    data_dir: Path = Path("data")
    results_dir: Path = Path("results")
    output_csv: Path = Path("results/elm_dataset_benchmark.csv")
    summary_csv: Path = Path("results/elm_dataset_benchmark_summary.csv")
    paired_csv: Path = Path("results/elm_dataset_paired_solver_comparisons.csv")
    datasets: List[str] = field(default_factory=lambda: ["MNIST", "Fashion-MNIST", "UCI HAR", "ISOLET"])
    methods: List[str] = field(default_factory=lambda: ["svd_default", "svd_jacobian", "newton_schulz", "kaur", "sheikhi"])
    hidden_sizes: List[int] = field(default_factory=lambda: [128, 256, 512])
    activations: List[str] = field(default_factory=lambda: ["tanh", "sigmoid", "relu", "elu"])
    hidden_initializations: List[str] = field(default_factory=lambda: ["gaussian", "xavier", "he"])
    seeds: List[int] = field(default_factory=lambda: list(range(10)))
    max_train_sample: int | None = 10000
    max_test_sample: int | None = 2000
    noise_levels: List[float] = field(default_factory=lambda: [0.0, 0.01, 0.05, 0.1])
    adam_max_epochs: List[int] = field(default_factory=lambda: [5, 10, 20, 50, 100])
    adam_learning_rate: float = 1e-2
    adam_weight_decay: float = 0.0
    pinv_max_iter: int = 50
    pinv_residual_tolerance: float = 1e-12
    pinv_convergence_threshold: float = 1e-12
    pinv_max_residual: float | None = 1e6
    pinv_residual_function: str = "identity"
    pinv_residual_growth_failure_factor: float | None = 1.2
    pinv_svd_default_runtime_failure_multiplier: float | None = 10.0
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype: torch.dtype = torch.float64
    verbose: bool = True


@dataclass
class ControlledIllConditioningConfig(DatasetExperimentConfig):
    output_csv: Path = Path("results/elm_controlled_ill_conditioning.csv")
    summary_csv: Path = Path("results/elm_controlled_ill_conditioning_summary.csv")
    datasets: List[str] = field(default_factory=lambda: ["MNIST"])
    methods: List[str] = field(default_factory=lambda: ["svd_default", "svd_jacobian", "newton_schulz", "kaur", "sheikhi"])
    hidden_sizes: List[int] = field(default_factory=lambda: [128, 256, 512, 1024])
    activations: List[str] = field(default_factory=lambda: ["tanh", "sigmoid"])
    hidden_initializations: List[str] = field(default_factory=lambda: ["xavier"])
    seeds: List[int] = field(default_factory=lambda: list(range(10)))
    saturation_scales: List[float] = field(default_factory=lambda: [1.0, 5.0, 10.0, 25.0, 50.0])
    duplicate_fractions: List[float] = field(default_factory=lambda: [0.0, 0.1, 0.25, 0.5])
    duplicate_noise: float = 1e-8


In [8]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch


def _ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def _set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _safe_ratio(num: torch.Tensor | float, den: torch.Tensor | float, eps: float = 1e-30) -> float:
    if isinstance(num, torch.Tensor):
        num_value = float(num.item())
    else:
        num_value = float(num)
    if isinstance(den, torch.Tensor):
        den_value = float(den.item())
    else:
        den_value = float(den)
    return num_value / max(abs(den_value), eps)


def _one_hot(labels: torch.Tensor, n_classes: int, dtype: torch.dtype) -> torch.Tensor:
    out = torch.zeros(labels.shape[0], n_classes, device=labels.device, dtype=dtype)
    out.scatter_(1, labels.long().unsqueeze(1), 1.0)
    return out


## Statistics

In [9]:
from __future__ import annotations

from typing import Sequence

import pandas as pd



def summarize_mean_std(
    df: pd.DataFrame,
    group_cols: Sequence[str],
    value_cols: Sequence[str],
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    numeric_cols = [col for col in value_cols if col in df.columns]
    summary_df = df[list(group_cols) + numeric_cols].copy()
    for col in numeric_cols:
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")
    grouped = summary_df.groupby(list(group_cols), dropna=False)[numeric_cols]
    summary = grouped.agg(["mean", "std", "count"]).reset_index()
    summary.columns = [
        "_".join([str(part) for part in col if part]).rstrip("_")
        if isinstance(col, tuple)
        else str(col)
        for col in summary.columns
    ]
    return summary


def summarize_dataset_results(df: pd.DataFrame, config: DatasetExperimentConfig) -> pd.DataFrame:
    value_cols = [
        "test_accuracy",
        "train_accuracy",
        "train_time_seconds",
        "training_success",
        "adam_max_epochs",
        "adam_final_loss",
        "pinv_final_convergence_delta",
        "hidden_condition",
        "hidden_rank",
        "beta_fro_norm",
        "training_residual",
        *[col for col in df.columns if col.startswith("noise_accuracy_")],
    ]
    summary = summarize_mean_std(
        df,
        group_cols=["dataset", "method", "hidden_size", "activation", "hidden_initialization"],
        value_cols=value_cols,
    )
    summary.to_csv(config.summary_csv, index=False)

    paired = paired_solver_comparisons(
        df,
        group_cols=["dataset", "hidden_size", "activation", "hidden_initialization"],
        metric="test_accuracy",
        baseline_method="svd_default",
    )
    paired.to_csv(config.paired_csv, index=False)
    return summary


def paired_solver_comparisons(
    df: pd.DataFrame,
    group_cols: Sequence[str],
    metric: str,
    baseline_method: str,
) -> pd.DataFrame:
    if df.empty or metric not in df.columns:
        return pd.DataFrame()
    rows = []
    index_cols = [*group_cols, "seed"]
    for group_key, group_df in df.groupby(list(group_cols), dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        pivot = group_df.pivot_table(index=index_cols, columns="method", values=metric, aggfunc="mean")
        if baseline_method not in pivot.columns:
            continue
        for method in pivot.columns:
            if method == baseline_method:
                continue
            deltas = (pivot[method] - pivot[baseline_method]).dropna()
            if deltas.empty:
                continue
            row = {col: value for col, value in zip(group_cols, group_key)}
            row.update(
                {
                    "metric": metric,
                    "baseline_method": baseline_method,
                    "method": method,
                    "mean_delta": float(deltas.mean()),
                    "std_delta": float(deltas.std(ddof=1)) if len(deltas) > 1 else 0.0,
                    "n_pairs": int(deltas.shape[0]),
                    "consistent_positive_fraction": float((deltas > 0).mean()),
                }
            )
            rows.append(row)
    return pd.DataFrame(rows)


## ELM Wrapper

In [10]:
from __future__ import annotations

import csv
import math
import time
from pathlib import Path
from typing import Callable

import pandas as pd
import torch
import torch.nn.functional as F


ADAM_METHOD_PREFIX = "adam_"


def adam_method_name(max_epochs: int) -> str:
    return f"{ADAM_METHOD_PREFIX}{max_epochs}_epochs"


def iter_elm_training_methods(config: DatasetExperimentConfig) -> list[str]:
    methods = list(config.methods)
    for max_epochs in config.adam_max_epochs:
        method = adam_method_name(max_epochs)
        if method not in methods:
            methods.append(method)
    return methods


def _adam_max_epochs_from_method(method: str) -> int | None:
    normalized = method.lower()
    if not normalized.startswith(ADAM_METHOD_PREFIX):
        return None
    suffix = normalized.removeprefix(ADAM_METHOD_PREFIX).removesuffix("_epochs")
    try:
        return int(suffix)
    except ValueError as exc:
        raise ValueError(f"Adam method names must look like 'adam_50_epochs', got '{method}'") from exc


def hidden_weight_tensor(
    init_method: str,
    hidden_size: int,
    input_size: int,
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    method = init_method.lower()
    if method in {"gaussian", "normal", "standard_normal"}:
        std = 1.0
    elif method in {"xavier", "glorot", "xavier_normal", "glorot_normal"}:
        std = math.sqrt(2.0 / (input_size + hidden_size))
    elif method in {"he", "kaiming", "he_normal", "kaiming_normal"}:
        std = math.sqrt(2.0 / input_size)
    elif method in {"lecun", "lecun_normal"}:
        std = math.sqrt(1.0 / input_size)
    else:
        raise ValueError(f"Unknown hidden initialization '{init_method}'")
    return std * torch.randn(hidden_size, input_size, device=device, dtype=dtype)


def apply_hidden_initialization(model: ExtremeLearningMachine, init_method: str) -> None:
    with torch.no_grad():
        model.input_weights.copy_(
            hidden_weight_tensor(
                init_method,
                model.hidden_size,
                model.input_size,
                model.device,
                model.dtype,
            )
        )
        model.biases.zero_()


def duplicate_hidden_neurons(model: ExtremeLearningMachine, duplicate_fraction: float, noise: float) -> None:
    if duplicate_fraction <= 0.0:
        return
    n_duplicate = min(model.hidden_size - 1, int(round(model.hidden_size * duplicate_fraction)))
    if n_duplicate <= 0:
        return
    with torch.no_grad():
        for idx in range(model.hidden_size - n_duplicate, model.hidden_size):
            source = idx % (model.hidden_size - n_duplicate)
            model.input_weights[idx].copy_(model.input_weights[source])
            model.biases[idx].copy_(model.biases[source])
            if noise > 0.0:
                model.input_weights[idx].add_(noise * torch.randn_like(model.input_weights[idx]))
                model.biases[idx].add_(noise * torch.randn_like(model.biases[idx]))


def scale_hidden_layer(model: ExtremeLearningMachine, scale: float) -> None:
    with torch.no_grad():
        model.input_weights.mul_(scale)
        model.biases.mul_(scale)


def _pinv_kwargs(config: DatasetExperimentConfig) -> dict[str, float | int | str]:
    return {
        "max_iter": config.pinv_max_iter,
        "residual_tolerance": config.pinv_residual_tolerance,
        "convergence_threshold": config.pinv_convergence_threshold,
        "max_residual": config.pinv_max_residual,
        "residual_function": config.pinv_residual_function,
        "residual_growth_failure_factor": config.pinv_residual_growth_failure_factor,
        "svd_default_runtime_failure_multiplier": config.pinv_svd_default_runtime_failure_multiplier,
    }


def _build_elm(
    input_size: int,
    method: str,
    hidden_size: int,
    activation: str,
    hidden_initialization: str,
    config: DatasetExperimentConfig,
    mutator: Callable[[ExtremeLearningMachine], None] | None = None,
) -> ExtremeLearningMachine:
    pinv_method = "svd_default" if _adam_max_epochs_from_method(method) is not None else method
    model = ExtremeLearningMachine(
        input_size=input_size,
        hidden_size=hidden_size,
        device=config.device,
        dtype=config.dtype,
        pinv_method=pinv_method,
        pinv_kwargs=_pinv_kwargs(config),
        activation=activation,
    )
    apply_hidden_initialization(model, hidden_initialization)
    if mutator is not None:
        mutator(model)
    return model


def _train_output_weights_with_adam(
    model: ExtremeLearningMachine,
    H_train: torch.Tensor,
    targets_train: torch.Tensor,
    max_epochs: int,
    config: DatasetExperimentConfig,
) -> dict[str, float | int | str | bool]:
    beta = torch.zeros(
        H_train.shape[1],
        targets_train.shape[1],
        device=H_train.device,
        dtype=H_train.dtype,
        requires_grad=True,
    )
    optimizer = torch.optim.Adam(
        [beta],
        lr=config.adam_learning_rate,
        weight_decay=config.adam_weight_decay,
    )
    loss = torch.tensor(float("nan"), device=H_train.device, dtype=H_train.dtype)
    for _ in range(max_epochs):
        optimizer.zero_grad(set_to_none=True)
        scores = H_train @ beta
        loss = F.mse_loss(scores, targets_train)
        loss.backward()
        optimizer.step()

    model.output_weights = beta.detach()
    final_scores = H_train @ model.output_weights
    final_loss = F.mse_loss(final_scores, targets_train).item()
    return {
        "optimizer": "adam",
        "adam_max_epochs": max_epochs,
        "adam_learning_rate": config.adam_learning_rate,
        "adam_weight_decay": config.adam_weight_decay,
        "iterations": max_epochs,
        "final_loss": final_loss,
        "success": bool(torch.isfinite(torch.as_tensor(final_loss))),
    }


def _fit_elm(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    n_classes: int,
    method: str,
    hidden_size: int,
    activation: str,
    hidden_initialization: str,
    seed: int,
    config: DatasetExperimentConfig,
    mutator: Callable[[ExtremeLearningMachine], None] | None = None,
) -> tuple[ExtremeLearningMachine, dict, float, torch.Tensor, torch.Tensor, torch.Tensor]:
    _set_seed(seed)
    targets_train = _one_hot(y_train, n_classes, config.dtype)
    adam_max_epochs = _adam_max_epochs_from_method(method)
    model = _build_elm(
        X_train.shape[1],
        method,
        hidden_size,
        activation,
        hidden_initialization,
        config,
        mutator=mutator,
    )

    start = time.time()
    if adam_max_epochs is None:
        train_info = model.train(X_train, targets_train)
    else:
        H_for_adam = model._compute_hidden_layer(X_train)
        train_info = _train_output_weights_with_adam(
            model,
            H_for_adam,
            targets_train,
            adam_max_epochs,
            config,
        )
    elapsed = time.time() - start
    with torch.no_grad():
        H_train = model._compute_hidden_layer(X_train)
        train_scores = H_train @ model.output_weights
    return model, train_info, elapsed, H_train, train_scores, targets_train


def _rank_from_svdvals(s: torch.Tensor) -> int:
    if s.numel() == 0:
        return 0
    tol = max(s.shape) * torch.finfo(s.dtype).eps * s.max().item()
    return int((s > tol).sum().item())


def _hidden_statistics(H: torch.Tensor, beta: torch.Tensor, targets: torch.Tensor) -> dict[str, float | int]:
    with torch.no_grad():
        s = torch.linalg.svdvals(H)
        sigma_min = s[-1].item() if s.numel() else float("nan")
        sigma_max = s[0].item() if s.numel() else float("nan")
        kappa = sigma_max / sigma_min if sigma_min > 0.0 else float("inf")
        residual = _safe_ratio(
            torch.linalg.matrix_norm(H @ beta - targets, ord="fro"),
            torch.linalg.matrix_norm(targets, ord="fro"),
        )
        return {
            "hidden_condition": kappa,
            "hidden_rank": _rank_from_svdvals(s),
            "hidden_sigma_min": sigma_min,
            "hidden_sigma_max": sigma_max,
            "beta_fro_norm": torch.linalg.matrix_norm(beta, ord="fro").item(),
            "training_residual": residual,
        }


def train_evaluate_elm(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    n_classes: int,
    method: str,
    hidden_size: int,
    activation: str,
    hidden_initialization: str,
    seed: int,
    config: DatasetExperimentConfig,
    mutator: Callable[[ExtremeLearningMachine], None] | None = None,
) -> dict[str, float | int | str | bool]:
    model, train_info, elapsed, H_train, train_scores, targets_train = _fit_elm(
        X_train,
        y_train,
        n_classes,
        method,
        hidden_size,
        activation,
        hidden_initialization,
        seed,
        config,
        mutator=mutator,
    )
    with torch.no_grad():
        test_scores = model.predict(X_test)

    return {
        "method": method,
        "hidden_size": hidden_size,
        "activation": activation,
        "hidden_initialization": hidden_initialization,
        "seed": seed,
        "train_accuracy": accuracy_from_scores(train_scores, y_train),
        "test_accuracy": accuracy_from_scores(test_scores, y_test),
        "train_time_seconds": elapsed,
        "optimizer": train_info.get("optimizer", "pinv"),
        "adam_max_epochs": train_info.get("adam_max_epochs"),
        "adam_learning_rate": train_info.get("adam_learning_rate"),
        "adam_weight_decay": train_info.get("adam_weight_decay"),
        "adam_final_loss": train_info.get("final_loss"),
        "training_success": bool(train_info.get("success", False)),
        "pinv_iterations": train_info.get("iterations", ""),
        "pinv_final_residual": train_info.get("final_residual"),
        "pinv_final_convergence_delta": train_info.get("final_convergence_delta"),
        "pinv_residual_function": train_info.get("residual_function", ""),
        "pinv_stopped_by": train_info.get("stopped_by", ""),
        "pinv_success": bool(train_info.get("success", False)) if train_info.get("optimizer", "pinv") == "pinv" else None,
        **classification_metrics_from_scores(test_scores, y_test, n_classes),
        **_hidden_statistics(H_train, model.output_weights, targets_train),
    }


def _noise_accuracy(
    model: ExtremeLearningMachine,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    noise_level: float,
    seed: int,
) -> float:
    if noise_level == 0.0:
        return accuracy_from_scores(model.predict(X_test), y_test)
    generator = torch.Generator(device=X_test.device)
    generator.manual_seed(seed)
    noisy = X_test + noise_level * torch.randn(X_test.shape, generator=generator, device=X_test.device, dtype=X_test.dtype)
    return accuracy_from_scores(model.predict(noisy), y_test)


def evaluate_noise_robustness(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    n_classes: int,
    method: str,
    hidden_size: int,
    activation: str,
    hidden_initialization: str,
    seed: int,
    config: DatasetExperimentConfig,
) -> dict[str, float]:
    model, *_ = _fit_elm(
        X_train,
        y_train,
        n_classes,
        method,
        hidden_size,
        activation,
        hidden_initialization,
        seed,
        config,
    )
    return {
        f"noise_accuracy_{noise_level:g}": _noise_accuracy(model, X_test, y_test, noise_level, seed + 99)
        for noise_level in config.noise_levels
    }


def _failed_elm_row(
    experiment: str,
    dataset_name: str,
    method: str,
    hidden_size: int,
    activation: str,
    hidden_initialization: str,
    seed: int,
    exc: Exception,
    **extra: float | int | str,
) -> dict[str, float | int | str | bool]:
    return {
        "experiment": experiment,
        "dataset": dataset_name,
        "method": method,
        "hidden_size": hidden_size,
        "activation": activation,
        "hidden_initialization": hidden_initialization,
        "seed": seed,
        "optimizer": "",
        "training_success": False,
        "pinv_success": False,
        "error": str(exc),
        **extra,
    }


def _append_progress(path: Path, row: dict, all_rows: list[dict]) -> None:
    """Keep a usable CSV if a long run is interrupted."""
    _ensure_dir(path.parent)
    fieldnames = sorted({key for item in all_rows for key in item.keys()})
    file_exists = path.exists()
    if not file_exists:
        with path.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerow({key: row.get(key, "") for key in fieldnames})
        return
    pd.DataFrame(all_rows).to_csv(path, index=False)


## Plots

In [11]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch

try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False


def _set_plot_theme() -> None:
    sns.set_theme(style="whitegrid", context="paper")


def _place_subplot_legend(ax, title: str) -> None:
    if ax.get_legend() is not None:
        ax.legend(
            fontsize=6,
            title=title,
            title_fontsize=6,
            loc="best",
            frameon=True,
            framealpha=0.85,
        )


def _two_column_axes(n_plots: int, width_per_col: float = 5.2, height_per_row: float = 3.5):
    n_cols = min(2, n_plots)
    n_rows = int(math.ceil(n_plots / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(width_per_col * n_cols, height_per_row * n_rows),
        squeeze=False,
        constrained_layout=True,
    )
    return fig, axes.reshape(-1), n_cols, n_rows


def _plot_synthetic_runtime_all_methods(df: pd.DataFrame, results_dir: Path) -> None:
    runtime_required = {"matrix_rows", "matrix_cols", "spectrum", "method", "runtime_seconds"}
    if not runtime_required.issubset(df.columns):
        return

    runtime_df = df.copy()
    runtime_df["runtime_seconds"] = pd.to_numeric(runtime_df["runtime_seconds"], errors="coerce")
    runtime_df = runtime_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["runtime_seconds"])
    if runtime_df.empty:
        return

    runtime_df["matrix_entries"] = pd.to_numeric(runtime_df["matrix_rows"], errors="coerce") * pd.to_numeric(
        runtime_df["matrix_cols"],
        errors="coerce",
    )
    runtime_df["matrix_size"] = runtime_df["matrix_rows"].astype(str) + "x" + runtime_df["matrix_cols"].astype(str)
    runtime_df = runtime_df.dropna(subset=["matrix_entries"]).sort_values(["matrix_entries", "matrix_rows", "matrix_cols"])
    size_order = runtime_df[["matrix_size", "matrix_entries"]].drop_duplicates().sort_values("matrix_entries")["matrix_size"]
    runtime_df["matrix_size"] = pd.Categorical(runtime_df["matrix_size"], categories=list(size_order), ordered=True)
    if runtime_df.empty:
        return

    spectra = list(runtime_df["spectrum"].drop_duplicates())
    fig, axes_flat, _, _ = _two_column_axes(len(spectra), width_per_col=4.8, height_per_row=3.4)

    for idx, spectrum in enumerate(spectra):
        ax = axes_flat[idx]
        spectrum_runtime = runtime_df[runtime_df["spectrum"] == spectrum]
        sns.lineplot(
            data=spectrum_runtime,
            x="matrix_size",
            y="runtime_seconds",
            hue="method",
            marker="o",
            estimator="mean",
            errorbar="sd",
            ax=ax,
        )
        ax.set_yscale("log")
        ax.set_title(str(spectrum), fontsize=9)
        ax.set_xlabel("matrix size")
        ax.set_ylabel("runtime (s)")
        ax.tick_params(axis="x", rotation=35)
        _place_subplot_legend(ax, "method")

    for ax in axes_flat[len(spectra):]:
        ax.axis("off")

    fig.savefig(results_dir / "synthetic_matrix_runtime_all_methods.png", dpi=150)
    plt.close(fig)


def plot_synthetic_matrix_results(df: pd.DataFrame, results_dir: Path) -> None:
    if not HAS_PLOTTING or df.empty:
        return
    _set_plot_theme()
    _ensure_dir(results_dir)
    _plot_synthetic_runtime_all_methods(df, results_dir)

    valid = df[df.get("success", False) == True].copy()
    if valid.empty:
        return

    valid["condition_number"] = pd.to_numeric(valid["condition_number"], errors="coerce")
    valid = valid[np.isfinite(valid["condition_number"])]
    valid = valid[valid["condition_number"] > 0]
    if valid.empty:
        return

    plot_specs = [
        ("relative_residual", "relative residual"),
        ("penrose_1_relative", "Penrose 1 residual"),
        ("runtime_seconds", "runtime (s)"),
    ]
    fig, axes, _, _ = _two_column_axes(len(plot_specs), width_per_col=5.2, height_per_row=3.8)

    for ax, (metric, ylabel) in zip(axes, plot_specs):
        sns.lineplot(
            data=valid,
            x="condition_number",
            y=metric,
            hue="method",
            marker="o",
            estimator="mean",
            errorbar="sd",
            ax=ax,
        )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("condition number")
        ax.set_ylabel(ylabel)
        _place_subplot_legend(ax, "method")
    for ax in axes[len(plot_specs):]:
        ax.axis("off")
    fig.savefig(results_dir / "synthetic_matrix_accuracy_runtime.png", dpi=150)
    plt.close(fig)


def plot_dataset_results(df: pd.DataFrame, results_dir: Path, prefix: str = "dataset") -> None:
    if not HAS_PLOTTING or df.empty:
        return
    _set_plot_theme()
    _ensure_dir(results_dir)
    required = {"hidden_condition", "beta_fro_norm", "test_accuracy", "train_time_seconds"}
    if not required.issubset(df.columns):
        return
    valid = df.copy()
    valid["log10_hidden_condition"] = np.log10(pd.to_numeric(valid["hidden_condition"], errors="coerce"))
    valid = valid.replace([np.inf, -np.inf], np.nan).dropna(subset=["log10_hidden_condition"])
    if valid.empty:
        return

    plot_specs = [
        ("beta_fro_norm", "||beta||_F"),
        ("test_accuracy", "test accuracy"),
        ("train_time_seconds", "runtime (s)"),
    ]
    fig, axes, _, _ = _two_column_axes(len(plot_specs), width_per_col=5.2, height_per_row=3.8)

    for ax, (metric, ylabel) in zip(axes, plot_specs):
        sns.scatterplot(
            data=valid,
            x="log10_hidden_condition",
            y=metric,
            hue="method",
            s=18,
            alpha=0.65,
            edgecolor=None,
            ax=ax,
        )
        ax.set_xlabel("log10 kappa(H)")
        ax.set_ylabel(ylabel)
        _place_subplot_legend(ax, "method")
    for ax in axes[len(plot_specs):]:
        ax.axis("off")
    fig.savefig(results_dir / f"{prefix}_kappa_vs_beta_accuracy_runtime.png", dpi=150)
    plt.close(fig)

    if "hidden_size" in valid.columns:
        width_specs = [
            ("test_accuracy", "test accuracy"),
            ("train_accuracy", "train accuracy"),
            ("train_time_seconds", "runtime (s)"),
        ]
        width_specs = [(metric, ylabel) for metric, ylabel in width_specs if metric in valid.columns]
        fig, axes, _, _ = _two_column_axes(len(width_specs), width_per_col=5.2, height_per_row=3.8)
        for ax, (metric, ylabel) in zip(axes, width_specs):
            sns.lineplot(
                data=valid,
                x="hidden_size",
                y=metric,
                hue="method",
                marker="o",
                estimator="mean",
                errorbar="sd",
                ax=ax,
            )
            ax.set_xlabel("hidden width")
            ax.set_ylabel(ylabel)
            _place_subplot_legend(ax, "method")
        for ax in axes[len(width_specs):]:
            ax.axis("off")
        fig.savefig(results_dir / f"{prefix}_hidden_width_seed_bands.png", dpi=150)
        plt.close(fig)


def plot_iterative_solver_trace(
    trace_data: pd.DataFrame | SyntheticMatrixBenchmarkConfig,
    results_dir: Path | None = None,
) -> None:
    if not HAS_PLOTTING:
        return
    _set_plot_theme()

    if isinstance(trace_data, SyntheticMatrixBenchmarkConfig):
        config = trace_data
        results_dir = config.results_dir
        if not config.residual_trace_csv.exists():
            return
        trace_df = pd.read_csv(config.residual_trace_csv)
    else:
        trace_df = trace_data.copy()

    if results_dir is None or trace_df.empty:
        return
    required = {"matrix_rows", "matrix_cols", "spectrum", "method", "iteration", "residual"}
    if not required.issubset(trace_df.columns):
        return
    _ensure_dir(results_dir)

    trace_df = trace_df.copy()
    trace_df["residual"] = pd.to_numeric(trace_df["residual"], errors="coerce")
    trace_df["iteration"] = pd.to_numeric(trace_df["iteration"], errors="coerce")
    trace_df = trace_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["residual", "iteration"])
    if trace_df.empty:
        return

    matrix_sizes = list(
        trace_df[["matrix_rows", "matrix_cols"]]
        .drop_duplicates()
        .sort_values(["matrix_rows", "matrix_cols"])
        .itertuples(index=False, name=None)
    )
    n_sizes = len(matrix_sizes)

    for spectrum, spectrum_df in trace_df.groupby("spectrum", sort=False):
        fig, axes_flat, n_cols, n_rows = _two_column_axes(n_sizes, width_per_col=4.2, height_per_row=3.0)

        any_trace = False
        for idx, (m, n) in enumerate(matrix_sizes):
            ax = axes_flat[idx]
            size_df = spectrum_df[(spectrum_df["matrix_rows"] == m) & (spectrum_df["matrix_cols"] == n)]

            if not size_df.empty:
                sns.lineplot(
                    data=size_df,
                    x="iteration",
                    y="residual",
                    hue="method",
                    marker="o",
                    markersize=2.5,
                    linewidth=1.2,
                    estimator="mean",
                    errorbar="sd",
                    ax=ax,
                )
                _place_subplot_legend(ax, "method")
                any_trace = True

            ax.set_yscale("log")
            ax.set_title(f"{m}x{n}", fontsize=9)
            if idx // n_cols == n_rows - 1:
                ax.set_xlabel("iteration")
            if idx % n_cols == 0:
                ax.set_ylabel("residual")

        for ax in axes_flat[n_sizes:]:
            ax.axis("off")

        if not any_trace:
            plt.close(fig)
            continue

        seed_label = ""
        if "seed" in spectrum_df.columns:
            n_seeds = spectrum_df["seed"].nunique(dropna=True)
            if n_seeds == 1:
                seed_label = f", seed={spectrum_df['seed'].iloc[0]}"
            elif n_seeds > 1:
                seed_label = f", {n_seeds} seeds"
        fig.suptitle(f"Iterative solver residual traces: {spectrum}{seed_label}", y=1.05)
        safe_spectrum = spectrum.lower().replace(" ", "_").replace("-", "_")
        fig.savefig(
            results_dir / f"iterative_solver_residual_traces_{safe_spectrum}.png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.close(fig)


def plot_representative_hidden_spectra(
    dataset_name: str,
    X_train: torch.Tensor,
    config: DatasetExperimentConfig,
) -> None:
    if not HAS_PLOTTING or not config.hidden_sizes or not config.activations or not config.hidden_initializations:
        return
    _set_plot_theme()
    _ensure_dir(config.results_dir)
    hidden_size = config.hidden_sizes[min(1, len(config.hidden_sizes) - 1)]
    init = config.hidden_initializations[0]
    seed = config.seeds[0] if config.seeds else 0
    _set_seed(seed)

    spectra_rows = []
    for activation in config.activations:
        model = _build_elm(
            X_train.shape[1],
            config.methods[0] if config.methods else "svd_default",
            hidden_size,
            activation,
            init,
            config,
        )
        with torch.no_grad():
            values = torch.linalg.svdvals(model._compute_hidden_layer(X_train)).detach().cpu().numpy()
        if values.size:
            spectra_rows.extend(
                {"index": index, "singular_value": value, "activation": activation}
                for index, value in enumerate(values, start=1)
            )

    if not spectra_rows:
        return
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    sns.lineplot(
        data=pd.DataFrame(spectra_rows),
        x="index",
        y="singular_value",
        hue="activation",
        errorbar=None,
        ax=ax,
    )
    ax.set_yscale("log")
    ax.set_xlabel("singular value index")
    ax.set_ylabel("singular value")
    ax.set_title(f"{dataset_name}: representative hidden spectra, L={hidden_size}, init={init}")
    _place_subplot_legend(ax, "activation")
    safe_name = dataset_name.lower().replace(" ", "_").replace("-", "_")
    fig.savefig(config.results_dir / f"{safe_name}_hidden_singular_spectra.png", dpi=150)
    plt.close(fig)


## Exp 1: Synthetic Matrix

In [12]:
from __future__ import annotations

import math
import time

import pandas as pd
import torch

def _qr_orthogonal(rows: int, cols: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    q, _ = torch.linalg.qr(torch.randn(rows, cols, device=device, dtype=dtype), mode="reduced")
    return q


def singular_values_for_spectrum(
    spectrum: str,
    rank_dim: int,
    dtype: torch.dtype,
    device: torch.device,
    zero_fraction: float = 0.1,
    small_cluster_fraction: float = 0.1,
) -> torch.Tensor:
    """
    Build the singular-value spectra requested in experiments.md.

    The markdown uses forms like 1^{-6}; here that is interpreted as the
    standard numerical-analysis notation 10^{-6}.
    """
    name = spectrum.lower()
    if name == "well_conditioned":
        return torch.linspace(1.0, 1e-1, rank_dim, device=device, dtype=dtype)
    if name == "moderately_ill_conditioned":
        return torch.logspace(0.0, -6.0, rank_dim, device=device, dtype=dtype)
    if name == "severely_ill_conditioned":
        return torch.logspace(0.0, -12.0, rank_dim, device=device, dtype=dtype)
    if name == "rank_deficient":
        values = torch.ones(rank_dim, device=device, dtype=dtype)
        n_zero = max(1, int(math.ceil(rank_dim * zero_fraction)))
        values[:-n_zero] = torch.linspace(1.0, 1e-3, rank_dim - n_zero, device=device, dtype=dtype)
        values[-n_zero:] = 0.0
        return values
    if name == "clustered_small":
        values = torch.linspace(1.0, 1e-3, rank_dim, device=device, dtype=dtype)
        n_cluster = max(2, int(math.ceil(rank_dim * small_cluster_fraction)))
        tiny = torch.finfo(dtype).eps * 10.0
        values[-n_cluster:] = tiny * torch.linspace(1.0, 10.0, n_cluster, device=device, dtype=dtype)
        return values
    raise ValueError(f"Unknown spectrum '{spectrum}'")


def generate_matrix_with_spectrum(
    rows: int,
    cols: int,
    spectrum: str,
    seed: int,
    config: SyntheticMatrixBenchmarkConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    _set_seed(seed)
    rank_dim = min(rows, cols)
    singular_values = singular_values_for_spectrum(
        spectrum,
        rank_dim,
        config.dtype,
        config.device,
        config.rank_deficient_zero_fraction,
        config.small_cluster_fraction,
    )
    u = _qr_orthogonal(rows, rank_dim, config.device, config.dtype)
    v = _qr_orthogonal(cols, rank_dim, config.device, config.dtype)
    matrix = (u * singular_values.unsqueeze(0)) @ v.t()
    return matrix, singular_values


def _penrose_metrics(A: torch.Tensor, X: torch.Tensor) -> dict[str, float]:
    a_norm = torch.linalg.matrix_norm(A, ord="fro")
    x_norm = torch.linalg.matrix_norm(X, ord="fro")
    return {
        "penrose_1_relative": _safe_ratio(torch.linalg.matrix_norm(A @ X @ A - A, ord="fro"), a_norm),
        "penrose_2_relative": _safe_ratio(torch.linalg.matrix_norm(X @ A @ X - X, ord="fro"), x_norm),
    }


def _matrix_rank_and_condition(singular_values: torch.Tensor) -> tuple[int, float]:
    if singular_values.numel() == 0:
        return 0, float("nan")
    tol = max(singular_values.shape) * torch.finfo(singular_values.dtype).eps * singular_values.max().item()
    rank = int((singular_values > tol).sum().item())
    nonzero = singular_values[singular_values > tol]
    if nonzero.numel() == 0:
        return rank, float("inf")
    return rank, float(nonzero.max().item() / nonzero.min().item())


def _residual_trace_rows(
    info: dict,
    matrix_rows: int,
    matrix_cols: int,
    spectrum: str,
    seed: int,
    method: str,
) -> list[dict[str, float | int | str]]:
    history = info.get("residual_history") or []
    return [
        {
            "matrix_rows": matrix_rows,
            "matrix_cols": matrix_cols,
            "spectrum": spectrum,
            "seed": seed,
            "method": method,
            "iteration": iteration,
            "residual": residual,
        }
        for iteration, residual in enumerate(history)
    ]


def run_synthetic_matrix_benchmark(
    config: SyntheticMatrixBenchmarkConfig = SyntheticMatrixBenchmarkConfig(),
) -> pd.DataFrame:
    """Run Experiment A and save raw and summary CSV files."""

    _ensure_dir(config.results_dir)
    rows: list[dict[str, float | int | str | bool]] = []
    trace_rows: list[dict[str, float | int | str]] = []

    for m, n in config.matrix_sizes:
        for spectrum in config.spectra:
            for seed in config.seeds:
                A, s = generate_matrix_with_spectrum(m, n, spectrum, seed, config)
                rank, kappa = _matrix_rank_and_condition(s)
                generator = torch.Generator(device=config.device)
                generator.manual_seed(seed + 10_000)
                B = torch.randn(m, config.target_cols, generator=generator, device=config.device, dtype=config.dtype)

                if config.verbose:
                    print(f"Synthetic A={m}x{n}, spectrum={spectrum}, seed={seed}, rank={rank}, kappa={kappa:.3e}")

                for method in config.methods:
                    start = time.time()
                    try:
                        pinv = Pseudoinverse(
                            device=config.device,
                            dtype=config.dtype,
                            method=method,
                            max_iter=config.pinv_max_iter,
                            residual_tolerance=config.pinv_residual_tolerance,
                            convergence_threshold=config.pinv_convergence_threshold,
                            max_residual=config.pinv_max_residual,
                            residual_function=config.pinv_residual_function,
                        )
                        info = pinv.pinv(A, return_info=True)
                        X_pinv = info["pinv"]
                        solution = X_pinv @ B
                        elapsed = time.time() - start
                        residual = torch.linalg.matrix_norm(A @ solution - B, ord="fro")
                        row = {
                            "experiment": "A",
                            "matrix_rows": m,
                            "matrix_cols": n,
                            "spectrum": spectrum,
                            "seed": seed,
                            "method": method,
                            "rank": rank,
                            "condition_number": kappa,
                            "relative_residual": _safe_ratio(residual, torch.linalg.matrix_norm(B, ord="fro")),
                            "runtime_seconds": elapsed,
                            "iterations": info.get("iterations", ""),
                            "solver_residual": info.get("final_residual", ""),
                            "solver_convergence_delta": info.get("final_convergence_delta", ""),
                            "residual_function": info.get("residual_function", ""),
                            "stopped_by": info.get("stopped_by", ""),
                            "success": bool(info.get("success", False)),
                            **_penrose_metrics(A, X_pinv),
                        }
                        if method not in SVD_METHODS:
                            trace_rows.extend(_residual_trace_rows(info, m, n, spectrum, seed, method))
                    except Exception as exc:
                        row = {
                            "experiment": "A",
                            "matrix_rows": m,
                            "matrix_cols": n,
                            "spectrum": spectrum,
                            "seed": seed,
                            "method": method,
                            "rank": rank,
                            "condition_number": kappa,
                            "relative_residual": float("inf"),
                            "runtime_seconds": time.time() - start,
                            "iterations": "",
                            "solver_residual": float("inf"),
                            "solver_convergence_delta": float("inf"),
                            "penrose_1_relative": float("inf"),
                            "penrose_2_relative": float("inf"),
                            "residual_function": config.pinv_residual_function,
                            "stopped_by": "",
                            "success": False,
                            "error": str(exc),
                        }
                    rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(config.output_csv, index=False)
    trace_df = pd.DataFrame(trace_rows)
    if not trace_df.empty:
        trace_df.to_csv(config.residual_trace_csv, index=False)
    summary = summarize_mean_std(
        df,
        group_cols=["matrix_rows", "matrix_cols", "spectrum", "method"],
        value_cols=["relative_residual", "penrose_1_relative", "penrose_2_relative", "runtime_seconds"],
    )
    summary.to_csv(config.summary_csv, index=False)
    plot_synthetic_matrix_results(df, config.results_dir)
    plot_iterative_solver_trace(trace_df, config.results_dir)
    return df


### Run

In [13]:
synthetic_result_dir = Path("/kaggle/working/") / "synthetic_matrix_benchmark_e7"
synthetic_config = SyntheticMatrixBenchmarkConfig(
    results_dir=synthetic_result_dir,
    output_csv=synthetic_result_dir / "results.csv",
    summary_csv=synthetic_result_dir / "summary.csv",
    residual_trace_csv=synthetic_result_dir / "residual_traces.csv",
    methods=PINV_METHODS,
    matrix_sizes=MATRIX_SIZES,
    pinv_residual_function='combined_penrose',
    pinv_residual_tolerance=None,
    pinv_convergence_threshold=1e-7,
    pinv_max_residual=1e3,
    verbose=True,
    seeds=[i * 11 + i + 7 for i in range(1)],
)
run_synthetic_matrix_benchmark(synthetic_config)

Synthetic A=100x100, spectrum=well_conditioned, seed=7, rank=100, kappa=1.000e+01
Synthetic A=100x100, spectrum=moderately_ill_conditioned, seed=7, rank=100, kappa=1.000e+06
Synthetic A=100x100, spectrum=severely_ill_conditioned, seed=7, rank=100, kappa=1.000e+12
Synthetic A=100x100, spectrum=rank_deficient, seed=7, rank=90, kappa=1.000e+03
Synthetic A=100x100, spectrum=clustered_small, seed=7, rank=90, kappa=9.813e+00
Synthetic A=500x500, spectrum=well_conditioned, seed=7, rank=500, kappa=1.000e+01
Synthetic A=500x500, spectrum=moderately_ill_conditioned, seed=7, rank=500, kappa=1.000e+06
Synthetic A=500x500, spectrum=severely_ill_conditioned, seed=7, rank=500, kappa=1.000e+12
Synthetic A=500x500, spectrum=rank_deficient, seed=7, rank=450, kappa=1.000e+03
Synthetic A=500x500, spectrum=clustered_small, seed=7, rank=450, kappa=9.891e+00
Synthetic A=500x200, spectrum=well_conditioned, seed=7, rank=200, kappa=1.000e+01
Synthetic A=500x200, spectrum=moderately_ill_conditioned, seed=7, rank

,experiment,matrix_rows,matrix_cols,spectrum,seed,method,rank,condition_number,relative_residual,runtime_seconds,iterations,solver_residual,solver_convergence_delta,residual_function,stopped_by,success,penrose_1_relative,penrose_2_relative
0,A,100,100,well_conditioned,7,svd_default,100,10.000000,3.327141e-15,0.501355,1,2.719943e-15,0.000000e+00,combined_penrose,direct,True,2.587548e-15,2.719943e-15
1,A,100,100,well_conditioned,7,svd_jacobian,100,10.000000,3.327141e-15,0.096241,1,2.719943e-15,0.000000e+00,combined_penrose,direct,True,2.587548e-15,2.719943e-15
2,A,100,100,well_conditioned,7,newton_schulz,100,10.000000,1.083594e-15,0.101617,17,9.450683e-16,4.551851e-10,combined_penrose,convergence,True,9.450683e-16,7.659440e-16
3,A,100,100,well_conditioned,7,chebyshev,100,10.000000,1.104936e-15,0.024140,11,9.166789e-16,3.398510e-09,combined_penrose,convergence,True,9.166789e-16,7.542269e-16
4,A,100,100,well_conditioned,7,homeier,100,10.000000,1.108489e-15,0.024640,10,9.260709e-16,7.484568e-10,combined_penrose,convergence,True,9.260709e-16,7.508526e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,A,300,1000,clustered_small,7,chebyshev,270,9.878093,3.341198e-01,0.136101,12,9.129983e-12,6.642220e-11,combined_penrose,convergence,True,7.507088e-15,9.129983e-12
236,A,300,1000,clustered_small,7,homeier,270,9.878093,3.341198e-01,0.129227,11,1.658728e-11,1.163836e-11,combined_penrose,convergence,True,7.495278e-15,1.658728e-11
237,A,300,1000,clustered_small,7,esmaeili,270,9.878093,3.341198e-01,0.101974,8,7.393824e-10,6.448351e-10,combined_penrose,convergence,True,1.126249e-14,7.393824e-10
238,A,300,1000,clustered_small,7,kaur,270,9.878093,3.341198e-01,0.109018,9,3.354757e-11,2.633196e-11,combined_penrose,convergence,True,7.928304e-15,3.354757e-11


In [14]:
synthetic_result_dir = Path("/kaggle/working/") / "synthetic_matrix_benchmark_e10"
synthetic_config = SyntheticMatrixBenchmarkConfig(
    results_dir=synthetic_result_dir,
    output_csv=synthetic_result_dir / "results.csv",
    summary_csv=synthetic_result_dir / "summary.csv",
    residual_trace_csv=synthetic_result_dir / "residual_traces.csv",
    methods=PINV_METHODS,
    matrix_sizes=MATRIX_SIZES,
    pinv_residual_function='combined_penrose',
    pinv_residual_tolerance=None,
    pinv_convergence_threshold=1e-10,
    pinv_max_residual=1e3,
    verbose=True,
    seeds=[i * 11 + i + 7 for i in range(1)],
)
run_synthetic_matrix_benchmark(synthetic_config)

Synthetic A=100x100, spectrum=well_conditioned, seed=7, rank=100, kappa=1.000e+01
Synthetic A=100x100, spectrum=moderately_ill_conditioned, seed=7, rank=100, kappa=1.000e+06
Synthetic A=100x100, spectrum=severely_ill_conditioned, seed=7, rank=100, kappa=1.000e+12
Synthetic A=100x100, spectrum=rank_deficient, seed=7, rank=90, kappa=1.000e+03
Synthetic A=100x100, spectrum=clustered_small, seed=7, rank=90, kappa=9.813e+00
Synthetic A=500x500, spectrum=well_conditioned, seed=7, rank=500, kappa=1.000e+01
Synthetic A=500x500, spectrum=moderately_ill_conditioned, seed=7, rank=500, kappa=1.000e+06
Synthetic A=500x500, spectrum=severely_ill_conditioned, seed=7, rank=500, kappa=1.000e+12
Synthetic A=500x500, spectrum=rank_deficient, seed=7, rank=450, kappa=1.000e+03
Synthetic A=500x500, spectrum=clustered_small, seed=7, rank=450, kappa=9.891e+00
Synthetic A=500x200, spectrum=well_conditioned, seed=7, rank=200, kappa=1.000e+01
Synthetic A=500x200, spectrum=moderately_ill_conditioned, seed=7, rank

,experiment,matrix_rows,matrix_cols,spectrum,seed,method,rank,condition_number,relative_residual,runtime_seconds,iterations,solver_residual,solver_convergence_delta,residual_function,stopped_by,success,penrose_1_relative,penrose_2_relative
0,A,100,100,well_conditioned,7,svd_default,100,10.000000,3.327141e-15,0.124796,1,2.719943e-15,0.000000e+00,combined_penrose,direct,True,2.587548e-15,2.719943e-15
1,A,100,100,well_conditioned,7,svd_jacobian,100,10.000000,3.327141e-15,0.118075,1,2.719943e-15,0.000000e+00,combined_penrose,direct,True,2.587548e-15,2.719943e-15
2,A,100,100,well_conditioned,7,newton_schulz,100,10.000000,1.035059e-15,0.027992,18,7.324111e-16,7.341677e-16,combined_penrose,convergence,True,7.237675e-16,7.324111e-16
3,A,100,100,well_conditioned,7,chebyshev,100,10.000000,9.657197e-16,0.021564,12,7.252009e-16,7.200022e-16,combined_penrose,convergence,True,7.252009e-16,7.138509e-16
4,A,100,100,well_conditioned,7,homeier,100,10.000000,1.020455e-15,0.021748,11,7.340748e-16,7.341079e-16,combined_penrose,convergence,True,7.075914e-16,7.340748e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,A,300,1000,clustered_small,7,chebyshev,270,9.878093,3.341198e-01,0.142754,12,9.129983e-12,6.642220e-11,combined_penrose,convergence,True,7.507088e-15,9.129983e-12
236,A,300,1000,clustered_small,7,homeier,270,9.878093,3.341198e-01,0.135854,11,1.658728e-11,1.163836e-11,combined_penrose,convergence,True,7.495278e-15,1.658728e-11
237,A,300,1000,clustered_small,7,esmaeili,270,9.878093,1.150577e-02,0.598349,50,1.148147e-02,9.659327e-03,combined_penrose,,False,1.148147e-02,8.449780e-03
238,A,300,1000,clustered_small,7,kaur,270,9.878093,3.341198e-01,0.112357,9,3.354757e-11,2.633196e-11,combined_penrose,convergence,True,7.928304e-15,3.354757e-11


## Exp 2: ELM on Datasets

In [15]:
from __future__ import annotations

import numpy as np
import torch


try:
    from sklearn.datasets import fetch_openml
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, StandardScaler

    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

try:
    from torchvision import datasets, transforms

    HAS_TORCHVISION = True
except ImportError:
    HAS_TORCHVISION = False

try:
    from ucimlrepo import fetch_ucirepo

    HAS_UCIMLREPO = True
except ImportError:
    HAS_UCIMLREPO = False


def _limit_samples(X: np.ndarray, y: np.ndarray, limit: int | None, seed: int) -> tuple[np.ndarray, np.ndarray]:
    if limit is None or X.shape[0] <= limit:
        return X, y
    rng = np.random.default_rng(seed)
    indices = rng.choice(X.shape[0], size=limit, replace=False)
    return X[indices], y[indices]


def _torchvision_dataset(
    dataset_name: str,
    config: DatasetExperimentConfig,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, int]:
    if not HAS_TORCHVISION:
        raise ImportError("torchvision is required for MNIST and Fashion-MNIST.")

    dataset_cls = datasets.MNIST if dataset_name.lower() == "mnist" else datasets.FashionMNIST
    transform = transforms.ToTensor()
    train_ds = dataset_cls(root=str(config.data_dir), train=True, download=True, transform=transform)
    test_ds = dataset_cls(root=str(config.data_dir), train=False, download=True, transform=transform)

    X_train = train_ds.data.numpy().reshape(len(train_ds), -1).astype(np.float64) / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)
    X_test = test_ds.data.numpy().reshape(len(test_ds), -1).astype(np.float64) / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)
    X_train, y_train = _limit_samples(X_train, y_train, config.max_train_sample, seed)
    X_test, y_test = _limit_samples(X_test, y_test, config.max_test_sample, seed + 1)
    return _to_tensors(X_train, y_train, X_test, y_test, config), 10


def _load_uci_har_local(
    config: DatasetExperimentConfig,
    seed: int,
) -> tuple[tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor], int]:
    root = config.data_dir / "UCI HAR Dataset"
    train_x = root / "train" / "X_train.txt"
    train_y = root / "train" / "y_train.txt"
    test_x = root / "test" / "X_test.txt"
    test_y = root / "test" / "y_test.txt"
    if not all(path.exists() for path in [train_x, train_y, test_x, test_y]):
        raise FileNotFoundError(
            "UCI HAR files not found. Expected data/UCI HAR Dataset/{train,test}/X_*.txt and y_*.txt."
        )
    X_train = np.loadtxt(train_x, dtype=np.float64)
    y_train = np.loadtxt(train_y, dtype=np.int64) - 1
    X_test = np.loadtxt(test_x, dtype=np.float64)
    y_test = np.loadtxt(test_y, dtype=np.int64) - 1
    X_train, y_train = _limit_samples(X_train, y_train, config.max_train_sample, seed)
    X_test, y_test = _limit_samples(X_test, y_test, config.max_test_sample, seed + 1)
    return _to_tensors(X_train, y_train, X_test, y_test, config), 6


def _load_uci_har_ucimlrepo(
    config: DatasetExperimentConfig,
    seed: int,
) -> tuple[tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor], int]:
    if not HAS_UCIMLREPO:
        raise ImportError(
            "ucimlrepo is required to download UCI HAR when local files are missing. "
            "Install it with `pip install ucimlrepo`."
        )
    if not HAS_SKLEARN:
        raise ImportError("scikit-learn is required to split UCI HAR data loaded from ucimlrepo.")

    data = fetch_ucirepo(id=240)
    X = np.asarray(data.data.features, dtype=np.float64)
    labels = LabelEncoder().fit_transform(np.asarray(data.data.targets).ravel())
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        labels,
        test_size=0.3,
        random_state=seed,
        stratify=labels,
    )
    X_train, y_train = _limit_samples(X_train, y_train, config.max_train_sample, seed)
    X_test, y_test = _limit_samples(X_test, y_test, config.max_test_sample, seed + 1)
    return _to_tensors(X_train, y_train, X_test, y_test, config), int(np.max(labels) + 1)


def _load_isolet_openml(
    config: DatasetExperimentConfig,
    seed: int,
) -> tuple[tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor], int]:
    if not HAS_SKLEARN:
        raise ImportError("scikit-learn is required to load ISOLET from OpenML.")
    data = fetch_openml(name="isolet", version=1, as_frame=False, parser="auto")
    X = np.asarray(data.data, dtype=np.float64)
    labels = LabelEncoder().fit_transform(np.asarray(data.target))
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        labels,
        test_size=0.2,
        random_state=seed,
        stratify=labels,
    )
    X_train, y_train = _limit_samples(X_train, y_train, config.max_train_sample, seed)
    X_test, y_test = _limit_samples(X_test, y_test, config.max_test_sample, seed + 1)
    return _to_tensors(X_train, y_train, X_test, y_test, config), int(np.max(labels) + 1)


def _to_tensors(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    config: DatasetExperimentConfig,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    if HAS_SKLEARN:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    return (
        torch.as_tensor(X_train, device=config.device, dtype=config.dtype),
        torch.as_tensor(y_train, device=config.device, dtype=torch.long),
        torch.as_tensor(X_test, device=config.device, dtype=config.dtype),
        torch.as_tensor(y_test, device=config.device, dtype=torch.long),
    )


def load_dataset(
    dataset_name: str,
    config: DatasetExperimentConfig,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, int]:
    """Load one requested dataset as tensors."""
    normalized = dataset_name.strip().lower()
    if normalized == "mnist":
        tensors, n_classes = _torchvision_dataset("mnist", config, seed)
        return (*tensors, n_classes)
    if normalized in {"fashion-mnist", "fashion_mnist", "fashionmnist"}:
        tensors, n_classes = _torchvision_dataset("fashion-mnist", config, seed)
        return (*tensors, n_classes)
    if normalized in {"uci har", "uci_har", "har"}:
        try:
            tensors, n_classes = _load_uci_har_local(config, seed)
        except FileNotFoundError:
            tensors, n_classes = _load_uci_har_ucimlrepo(config, seed)
        return (*tensors, n_classes)
    if normalized == "isolet":
        tensors, n_classes = _load_isolet_openml(config, seed)
        return (*tensors, n_classes)
    raise ValueError(f"Unknown dataset '{dataset_name}'")


In [16]:
from __future__ import annotations

import pandas as pd


def run_dataset_experiments(config: DatasetExperimentConfig = DatasetExperimentConfig()) -> pd.DataFrame:
    """Run Experiment B over datasets, seeds, widths, activations, initializers, and solvers."""
    _ensure_dir(config.results_dir)
    rows: list[dict] = []
    methods = iter_elm_training_methods(config)
    total_runs = (
        len(config.datasets)
        * len(config.seeds)
        * len(config.hidden_sizes)
        * len(config.activations)
        * len(config.hidden_initializations)
        * len(methods)
    )
    run_index = 0

    for dataset_name in config.datasets:
        X_train, y_train, X_test, y_test, n_classes = load_dataset(dataset_name, config, seed=config.seeds[0])
        if config.verbose:
            print(f"Dataset {dataset_name}: train={tuple(X_train.shape)}, test={tuple(X_test.shape)}, classes={n_classes}")
        plot_representative_hidden_spectra(dataset_name, X_train, config)

        for seed in config.seeds:
            for hidden_size in config.hidden_sizes:
                for activation in config.activations:
                    for init in config.hidden_initializations:
                        for method in methods:
                            run_index += 1
                            if config.verbose:
                                print(
                                    "Running dataset experiment "
                                    f"{run_index}/{total_runs}: "
                                    f"dataset={dataset_name}, seed={seed}, hidden_size={hidden_size}, "
                                    f"activation={activation}, hidden_initialization={init}, method={method}",
                                    flush=True,
                                )
                            try:
                                row = train_evaluate_elm(
                                    X_train,
                                    y_train,
                                    X_test,
                                    y_test,
                                    n_classes,
                                    method,
                                    hidden_size,
                                    activation,
                                    init,
                                    seed,
                                    config,
                                )
                                row.update({"experiment": "B", "dataset": dataset_name})
                                noise_accs = evaluate_noise_robustness(
                                    X_train,
                                    y_train,
                                    X_test,
                                    y_test,
                                    n_classes,
                                    method,
                                    hidden_size,
                                    activation,
                                    init,
                                    seed,
                                    config,
                                )
                                row.update(noise_accs)
                            except Exception as exc:
                                row = _failed_elm_row(
                                    "B",
                                    dataset_name,
                                    method,
                                    hidden_size,
                                    activation,
                                    init,
                                    seed,
                                    exc,
                                )
                            rows.append(row)
                            _append_progress(config.output_csv, row, rows)

    df = pd.DataFrame(rows)
    df.to_csv(config.output_csv, index=False)
    summarize_dataset_results(df, config)
    plot_dataset_results(df, config.results_dir)
    return df


### Run

In [17]:
dataset_result_dir = Path("/kaggle/working/") / "dataset_benchmark"
datasets_config = DatasetExperimentConfig(
    results_dir=dataset_result_dir,
    output_csv=dataset_result_dir / "results.csv",
    summary_csv=dataset_result_dir / "summary.csv",
    paired_csv=dataset_result_dir / "paired_comparisons.csv",
    datasets=[ "mnist", "fashion_mnist", "isolet"],
    methods=PINV_METHODS,
    hidden_sizes=[128, 256, 512],
    activations=["relu", "elu", "sigmoid", "tanh"],
    # max_train_sample=10000,
    # max_test_sample=2000,
    pinv_residual_tolerance=None,
    pinv_residual_function='combined_penrose',
    pinv_convergence_threshold=1e-7,
    pinv_max_iter=30,
    pinv_max_residual=1e3,
    pinv_svd_default_runtime_failure_multiplier=2.0,
    adam_learning_rate=1e-3,
    adam_max_epochs=[5, 10, 20, 50, 100],
    adam_weight_decay=0.0,
    noise_levels=[0.0],
    
    verbose=True,
    seeds=[i * 11 + i + 7 for i in range(1)],
)
run_dataset_experiments(datasets_config)

100%|██████████| 9.91M/9.91M [00:00<00:00, 33.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.14MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.1MB/s]


Dataset mnist: train=(10000, 784), test=(2000, 784), classes=10
Running dataset experiment 1/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=svd_default
Running dataset experiment 2/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=svd_jacobian
Running dataset experiment 3/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=newton_schulz
Running dataset experiment 4/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=chebyshev
Running dataset experiment 5/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=homeier
Running dataset experiment 6/1404: dataset=mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=esmaeili
Running dataset experiment 7/1404: dataset=mnist, seed=7, hidden_size=128, activatio

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 272kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.05MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.1MB/s]


Dataset fashion_mnist: train=(10000, 784), test=(2000, 784), classes=10
Running dataset experiment 469/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=svd_default
Running dataset experiment 470/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=svd_jacobian
Running dataset experiment 471/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=newton_schulz
Running dataset experiment 472/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=chebyshev
Running dataset experiment 473/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=homeier
Running dataset experiment 474/1404: dataset=fashion_mnist, seed=7, hidden_size=128, activation=relu, hidden_initialization=gaussian, method=esmaeili
Running dataset 

,method,hidden_size,activation,hidden_initialization,seed,train_accuracy,test_accuracy,train_time_seconds,optimizer,adam_max_epochs,...,mcc,hidden_condition,hidden_rank,hidden_sigma_min,hidden_sigma_max,beta_fro_norm,training_residual,experiment,dataset,noise_accuracy_0
0,svd_default,128,relu,gaussian,7,0.804700,0.809000,0.288466,pinv,NaN,...,0.788544,17.604261,128,680.764210,11984.350766,0.036224,0.701088,B,mnist,0.809000
1,svd_jacobian,128,relu,gaussian,7,0.804700,0.809000,0.289145,pinv,NaN,...,0.788544,17.604261,128,680.764210,11984.350766,0.036224,0.701088,B,mnist,0.809000
2,newton_schulz,128,relu,gaussian,7,0.499100,0.488000,0.759276,pinv,NaN,...,0.464016,17.604261,128,680.764210,11984.350766,0.000442,0.984107,B,mnist,0.488000
3,chebyshev,128,relu,gaussian,7,0.505700,0.497500,9.204783,pinv,NaN,...,0.473596,17.604261,128,680.764210,11984.350766,0.000640,0.977577,B,mnist,0.497500
4,homeier,128,relu,gaussian,7,0.511800,0.501000,18.408864,pinv,NaN,...,0.476729,17.604261,128,680.764210,11984.350766,0.000733,0.974649,B,mnist,0.501000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1399,adam_5_epochs,512,tanh,he,7,0.850890,0.836538,0.037955,adam,5.0,...,0.831481,40.468324,512,13.205517,534.405135,0.315958,0.803082,B,isolet,0.836538
1400,adam_10_epochs,512,tanh,he,7,0.907327,0.903205,0.056124,adam,10.0,...,0.899527,40.468324,512,13.205517,534.405135,0.467213,0.737731,B,isolet,0.903205
1401,adam_20_epochs,512,tanh,he,7,0.928972,0.919872,0.093361,adam,20.0,...,0.916784,40.468324,512,13.205517,534.405135,0.644700,0.689458,B,isolet,0.919872
1402,adam_50_epochs,512,tanh,he,7,0.948373,0.926282,0.203906,adam,50.0,...,0.923424,40.468324,512,13.205517,534.405135,0.897544,0.636639,B,isolet,0.926282
